# DeepDiscover: streamflow simulation and calibration

This notebook trains a two-DU model with an atmospheric loss boundary. The synthetic record has known water balance equations, allowing a check of the simulated outlet and the numerical mass residual. Water quantities are in millimetres per day.

Install the package from the repository root with `python -m pip install -e ".[examples,test]"`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from deepdiscover import (
    CRCNet, DiscoverUnit, MultiDUModel, DemandLimitedSink,
    DeepDiscover, HydroDataset, hydro_metrics,
)

tf.keras.utils.set_random_seed(42)

## Generate a conservative reference series

The first storage produces fast flow and water transfer to the second storage. The second storage produces slow flow. Both storages retain water from one day to the next.

In [ ]:
rng = np.random.default_rng(42)
n_days = 650
time = np.arange(n_days)
p = rng.gamma(1.5, 3.0, n_days).astype('float32')
p[rng.random(n_days) < 0.55] = 0
pet = (1.0 + 0.4 * np.sin(2 * np.pi * time / 365)).astype('float32')
s1 = s2 = 0.0
q_true, storage_true = [], []
for rain, demand in zip(p, pet):
    available1 = s1 + rain
    loss = min(available1, 0.60 * demand)
    resource1 = available1 - loss
    q_fast = 0.25 * resource1
    transfer = 0.15 * resource1
    s1 = resource1 - q_fast - transfer
    available2 = s2 + transfer
    q_slow = 0.08 * available2
    s2 = available2 - q_slow
    q_true.append(q_fast + q_slow)
    storage_true.append([s1, s2])
q_true = np.asarray(q_true, 'float32')
storage_true = np.asarray(storage_true, 'float32')
print(f'Mean reference flow: {q_true.mean():.2f} mm/day')

## Define the model

The route from the first DU to the second DU is instantaneous. Each process has one destination. CRC input order is precipitation information, PET information, beginning-of-step storage, and (when present) incoming routed water.

In [ ]:
def make_model():
    loss_efficiency = CRCNet([1, 1], [5, 100], output='sigmoid')
    du0 = DiscoverUnit(
        process_nets=[
            CRCNet([1, -1, 1], [5, 3, 100]),
            CRCNet([1, -1, 1], [5, 3, 100]),
        ],
        context_indices=[(0, 1), (0, 1)],
        exchanges=[DemandLimitedSink(loss_efficiency, demand_index=1, context_indices=(0,))],
    )
    du1 = DiscoverUnit(
        process_nets=[CRCNet([1, -1, 1, 1], [5, 3, 100, 5])],
        context_indices=[(0, 1)], upstream_count=1,
    )
    return DeepDiscover(MultiDUModel([du0, du1], destinations=[['outlet', 1], ['outlet']]),
                        observations={'streamflow': 'outlet'})

dd = make_model()
data = HydroDataset(
    water=np.column_stack([p, np.zeros_like(p)]),
    drivers=np.column_stack([p, pet]),
    observations={'streamflow': q_true},
)
train = data.make_windows(warmup=40, score_length=80, stride=40, first_score=40, last_score=460)
validation = data.continuous_batch(start=460, stop=560, warmup=40)
test = data.continuous_batch(start=560, stop=n_days, warmup=40)
print('Training water shape:', train.water.shape)
print('Validation water shape:', validation.water.shape)

## Calibrate on training windows

The first 40 days of each window are simulated but carry zero calibration weight. The validation period does not update model parameters.

In [ ]:
history = dd.fit(train, validation, epochs=30, batch_size=5,
                 learning_rate=0.005, patience=8, verbose=1)
print('Last epoch:', history[-1])

## Evaluate the reserved period

This demonstration uses the test period for reporting after selecting the model on validation.

In [ ]:
scores = dd.evaluate(test)
print(scores['metrics']['streamflow'])
print('Maximum numerical mass residual:', scores['max_abs_mass_residual'])
predicted = dd.predict(test.water, test.drivers)['streamflow'].numpy()[0, 40:, 0]
plt.figure(figsize=(11, 3))
plt.plot(q_true[560:], label='Reference', linewidth=1)
plt.plot(predicted, label='DeepDiscover', linewidth=1)
plt.legend()
plt.xlabel('Days in test period')
plt.ylabel('Flow (mm/day)')
plt.tight_layout(); plt.show()

## Save the fitted parameters

The model-creation function must be retained with the saved parameters for later reloading.

In [ ]:
dd.save_checkpoint('results/streamflow_example', metadata={'case': 'two_du_synthetic'})
print('Saved results/streamflow_example')